<dev dir='rtl' style='text-align:right'>

# 🐼 راهنمای جامع و خلاصه Pandas

راهنمای مرجع سریع (Cheat Sheet) برای مهم‌ترین قابلیت‌های کتابخانه **Pandas** در پایتون، برای کار روزمره با داده.

## 📑 فهرست مطالب

1. [Series و DataFrame چیستند؟](#1-series-و-dataframe-چیستند)
2. [خواندن و نوشتن داده‌های جدولی](#2-خواندن-و-نوشتن-داده‌های-جدولی)
3. [انتخاب زیرمجموعه‌ای از داده (Subsetting)](#3-انتخاب-زیرمجموعه‌ای-از-داده-subsetting)
4. [مدیریت داده‌های گمشده (Missing Data)](#4-مدیریت-داده‌های-گمشده-missing-data)
5. [ساخت ستون‌های جدید از ستون‌های موجود](#5-ساخت-ستون‌های-جدید-از-ستون‌های-موجود)
6. [محاسبه آمار خلاصه (Summary Statistics)](#6-محاسبه-آمار-خلاصه-summary-statistics)
7. [گروه‌بندی داده‌ها (GroupBy)](#7-گروه‌بندی-داده‌ها-groupby)
8. [تغییر شکل جدول (Sort / Pivot / Melt)](#8-تغییر-شکل-جدول-sort--pivot--melt)
9. [ترکیب جداول (Join / Merge / Concat)](#9-ترکیب-جداول-join--merge--concat)
10. [داده‌های سری زمانی (Time Series)](#10-داده‌های-سری-زمانی-time-series)
11. [پردازش داده‌های متنی (Text/String)](#11-پردازش-داده‌های-متنی-textstring)
12. [رسم نمودار (Visualization)](#12-رسم-نمودار-visualization)

<dev>

In [ ]:
import pandas as pd
import numpy as np

---

## 1. Series و DataFrame چیستند؟

Pandas دو ساختار داده اصلی دارد:
- **Series**: آرایه یک‌بعدی و برچسب‌دار (شبیه یک ستون).
- **DataFrame**: جدول دوبعدی که از چند Series (ستون) تشکیل شده.

In [ ]:
# ساخت یک Series
s = pd.Series(np.random.randn(1000))
s[::2] = np.nan          # هر عضو با ایندکس زوج را NaN می‌کند
s.describe()             # خلاصه آماری (count, mean, std, min, 25%, 50%, 75%, max)

# ساخت یک DataFrame
df = pd.DataFrame(np.random.randn(1000, 5), columns=['a', 'b', 'c', 'd', 'e'])
df.head()                # ۵ سطر ابتدایی
df.describe()            # خلاصه آماری تمام ستون‌های عددی

### `describe()` روی داده‌های متنی و ترکیبی

In [ ]:
pd.Series(["a", "a", "a", "b", "b", np.nan, "b"]).describe()

df.describe(include="object")  # فقط ستون‌های متنی
df.describe(include="number")  # فقط ستون‌های عددی
df.describe(include="all")     # همه ستون‌ها

### `idxmin` / `idxmax`

معادل `argmin`/`argmax` در NumPy، اما به‌جای اندیس عددی، **برچسب** سطر/ستون را برمی‌گرداند.

In [ ]:
s = pd.Series(np.random.randn(5))
s.idxmin(), s.idxmax()

df = pd.DataFrame(np.random.randn(5, 3), columns=["A", "B", "C"])
df.idxmin(axis=0)   # برای هر ستون
df.idxmin(axis=1)   # برای هر سطر

---

## 2. خواندن و نوشتن داده‌های جدولی

In [ ]:
# خواندن
df = pd.read_csv("file.csv")
df = pd.read_excel("file.xlsx", engine="openpyxl")   # نیاز به pip install openpyxl
df = pd.read_excel("file.xls", engine="xlrd")        # نیاز به pip install xlrd

# نوشتن
df.to_csv("out.csv", index=False)
df.to_excel("out.xlsx", sheet_name="Sheet1", index=False)

### مشاهده و بررسی سریع

In [ ]:
df.head(8)      # ۸ سطر ابتدایی
df.tail(8)      # ۸ سطر انتهایی
df.shape        # (تعداد سطر، تعداد ستون)

### `assign()` — افزودن ستون محاسباتی بدون تغییر df اصلی

In [ ]:
df.assign(sepal_ratio=df["sepal width (cm)"] / df["sepal length (cm)"])

# یا با lambda (بهتر برای زنجیره‌ای کردن عملیات)
df.assign(sepal_ratio=lambda x: x["sepal width (cm)"] / x["sepal length (cm)"])

### `query()` — فیلتر کردن با syntax شبیه SQL

In [ ]:
df.query("`sepal length (cm)` > 5")

# ترکیب query + assign در یک زنجیره
df.query("`sepal length (cm)` > 5").assign(
    sepal_ratio=lambda x: x["sepal width (cm)"] / x["sepal length (cm)"],
    petal_ratio=lambda x: x["petal width (cm)"] / x["petal length (cm)"]
)

---

## 3. انتخاب زیرمجموعه‌ای از داده (Subsetting)

In [ ]:
df.set_index("PassengerId", inplace=True)   # تعیین ستون به عنوان ایندکس

# انتخاب ستون‌ها
df["Age"]                     # یک ستون به‌صورت Series
df[["Age", "Sex"]]             # چند ستون به‌صورت DataFrame

# انتخاب سطرها با شرط (Boolean Indexing)
above_35 = df[df["Age"] > 35]
class_23 = df[df["Pclass"].isin([2, 3])]
# معادل:
class_23 = df[(df["Pclass"] == 2) | (df["Pclass"] == 3)]

age_no_na = df[df["Age"].notna()]   # حذف سطرهایی که Age آن‌ها خالی است

### `loc` در مقابل `iloc`

| | بر اساس | مثال |
|---|---|---|
| `loc` | **برچسب** (label) | `df.loc[df["Age"] > 35, "Name"]` |
| `iloc` | **موقعیت عددی** (position) | `df.iloc[9:25, 2:5]` |

In [ ]:
adult_names = df.loc[df["Age"] > 35, "Name"]   # سطرهای فیلترشده، فقط ستون Name
df.iloc[9:25, 2:5]                              # سطر ۹ تا ۲۴، ستون ۲ تا ۴
df.iloc[0:3, 2] = "Ali"                         # مقداردهی مستقیم با iloc

---

## 4. مدیریت داده‌های گمشده (Missing Data)

In [ ]:
missing_values = ["n.a.", "NA", "na", 0]   # مقادیری که باید NaN در نظر گرفته شوند
df = pd.read_csv("file.csv", na_values=missing_values)

دو راهکار اصلی برای داده گمشده وجود دارد: **حذف** یا **پرکردن**.

### حذف سطرها (`dropna`)

In [ ]:
df.dropna(axis=0, inplace=True, how="all")  # فقط سطرهایی که تمام مقادیرشان NaN است حذف شوند
# how="any" → اگر حتی یک مقدار NaN داشته باشد، سطر حذف می‌شود

### پر کردن مقادیر (`fillna` / `interpolate`)

In [ ]:
df["Salary"].fillna(df["Salary"].mean(), inplace=True)   # با میانگین
df["Salary"].fillna(1111, inplace=True)                   # با مقدار ثابت
df["Salary"].fillna(method="ffill", inplace=True)         # با مقدار قبلی (forward fill)
df["Salary"].fillna(method="bfill", inplace=True)         # با مقدار بعدی (backward fill)
df["Salary"] = df["Salary"].interpolate(method="linear")  # درون‌یابی خطی: x1 = (x0 + x2) / 2

---

## 5. ساخت ستون‌های جدید از ستون‌های موجود

In [ ]:
df["New_Col"] = df["Age"] / df["Fare"]   # ستون جدید از ترکیب دو ستون دیگر

# تغییر نام ستون‌ها
df_renamed = df.rename(columns={"Ticket": "Bilit"})

---

## 6. محاسبه آمار خلاصه (Summary Statistics)

In [ ]:
df["Age"].mean()
df[["Age", "Fare"]].median()
df[["Age", "Fare"]].describe()

# محاسبه چند آماره دلخواه برای چند ستون به‌صورت هم‌زمان
df.agg({
    "Age":  ["min", "max", "median"],
    "Fare": ["min", "max", "std"]
})

### آمار به تفکیک گروه

In [ ]:
df[["Sex", "Age"]].groupby("Sex").mean()
df.groupby("Sex")["Age"].mean()
df.groupby("Sex").mean(numeric_only=True)   # فقط روی ستون‌های عددی

num_cols = df.select_dtypes(include=["number"]).columns
df.groupby("Sex")[num_cols].mean()

### `value_counts` در مقابل `groupby().count()` / `size()`

In [ ]:
df["Pclass"].value_counts()          # تعداد هر مقدار (پیش‌فرض: NaN نادیده گرفته می‌شود)
df.groupby("Pclass").count()         # تعداد مقادیر غیر NaN در هر ستون به تفکیک گروه
df.groupby("Pclass")["Pclass"].count()

> **نکته:** `size()` تعداد کل سطرها (شامل NaN) را برمی‌گرداند، اما `count()` فقط مقادیر غیر NaN را می‌شمارد.

---

## 7. گروه‌بندی داده‌ها (GroupBy)

In [ ]:
grouped = df.groupby("Year")
grouped.groups                       # دیکشنری {مقدار گروه: ایندکس سطرها}
df.groupby(["Year", "Team"]).groups  # گروه‌بندی چندسطحی

for name, group in grouped:
    print(name)
    print(group)

grouped.get_group(2014)              # فقط سطرهای مربوط به یک گروه خاص

### توابع تجمیعی (Aggregation)

In [ ]:
grouped["Points"].agg("mean")
grouped["Points"].agg(["mean", "sum", "std"])   # چند تابع هم‌زمان
grouped["Points"].count()                        # تعداد مقادیر غیر خالی
grouped["Points"].size()                         # تعداد کل سطرها

### تبدیل (Transform) و فیلتر (Filter) در سطح گروه

In [ ]:
grouped["Points"].transform(lambda x: x * 2)     # روی هر عضو گروه اعمال می‌شود، شکل خروجی حفظ می‌شود
grouped.filter(lambda x: len(x) >= 1)             # فقط گروه‌هایی که شرط را دارند نگه داشته می‌شوند

---

## 8. تغییر شکل جدول (Sort / Pivot / Melt)

### مرتب‌سازی

In [ ]:
df.sort_values(by="Age").head()
df.sort_values(by=["Pclass", "Age"], ascending=False).head()

### `pivot` — تبدیل مقادیر یک ستون به چند ستون جدید

In [ ]:
data = pd.read_csv("weather.csv", index_col="date.utc", parse_dates=True)
no2_sub = data[data["parameter"] == "no2"]

no2_sub.pivot(columns="location", values="value")

### `pivot_table` — مثل pivot، اما با امکان تجمیع (aggregation)

In [ ]:
data.pivot_table(values="value", index="location", columns="parameter", aggfunc="mean")
# معادل:
data.groupby(["parameter", "location"]).mean(numeric_only=True)

### `melt` — عملیات معکوسِ pivot (از حالت عریض به بلند)

In [ ]:
pivoted = no2_sub.pivot(columns="location", values="value").reset_index()
pivoted.melt(id_vars="date.utc", value_name="My Values", var_name="My_VAR")

---

## 9. ترکیب جداول (Join / Merge / Concat)

### `join` — بر اساس ایندکس

In [ ]:
df1.join(df2, lsuffix="_caller", rsuffix="_other")
df1.join(df2.set_index("key"), on="key")

### `merge` — بر اساس مقدار ستون (شبیه SQL JOIN)

In [ ]:
dfm1.merge(dfm2, left_on="lkey", right_on="rkey")               # inner (پیش‌فرض)
dfm1.merge(dfm2, left_on="lkey", right_on="rkey", how="left")
dfm1.merge(dfm2, left_on="lkey", right_on="rkey", how="right")

### `concat` — چسباندن چند DataFrame به هم

In [ ]:
pd.concat([df1, df2], axis=0, join="outer")   # چسباندن عمودی (سطرها زیر هم)
pd.concat([df1, df2], axis=1, join="outer")   # چسباندن افقی (ستون‌ها کنار هم)
pd.concat([df1, df2], axis=0, join="inner")   # فقط ستون‌های مشترک نگه داشته می‌شوند

| پارامتر | معنی |
|---|---|
| `axis=0` | چسباندن سطر به سطر (زیر هم) |
| `axis=1` | چسباندن ستون به ستون (کنار هم) |
| `join="inner"` | فقط مقادیر/ستون‌های مشترک |
| `join="outer"` | همه مقادیر/ستون‌ها، جاهای خالی با NaN پر می‌شوند |

---

## 10. داده‌های سری زمانی (Time Series)

In [ ]:
air_quality = pd.read_csv("weather.csv")
air_quality = air_quality.rename(columns={"date.utc": "datetime"})

# تبدیل رشته متنی به نوع datetime — پیش‌نیاز تمام عملیات زیر
air_quality["datetime"] = pd.to_datetime(air_quality["datetime"])

air_quality["datetime"].max() - air_quality["datetime"].min()   # بازه زمانی کل داده

# استخراج اجزای تاریخ با accessor به نام dt
air_quality["month"] = air_quality["datetime"].dt.month
air_quality.groupby([air_quality["datetime"].dt.weekday, "location"])["value"].mean()

### `resample` — تغییر بازه زمانی (نمونه‌برداری مجدد)

In [ ]:
no2 = air_quality.pivot_table(index="datetime", columns="location", values="value", aggfunc="mean")

no2.resample("D").mean()    # میانگین روزانه
no2.resample("ME").mean()   # میانگین ماهانه (Month End)
no2.resample("ME").max()    # بیشینه ماهانه

---

## 11. پردازش داده‌های متنی (Text/String)

تمام متدهای رشته‌ای از طریق accessor به نام **`str`** روی یک ستون متنی در دسترس‌اند.

In [ ]:
titanic["Name"].str.lower()                         # حروف کوچک
titanic["surname"] = titanic["Name"].str.split(",").str.get(0)   # استخراج نام خانوادگی

willi = titanic["Name"].str.contains("William")       # جستجوی رشته (خروجی بولی)
titanic[willi]

titanic["Name"].str.len().idxmax()                    # ایندکس طولانی‌ترین نام
titanic.loc[titanic["Name"].str.len().idxmax(), "Name"]

# جایگزینی مقادیر متنی
titanic["Sex_short"] = titanic["Sex"].replace({"male": "M", "female": "F"})

---

## 12. رسم نمودار (Visualization)

In [ ]:
import matplotlib.pyplot as plt

Pandas یک متد `.plot()` روی DataFrame/Series دارد که مستقیماً روی matplotlib کار می‌کند.

In [ ]:
# نمودار خطی
df.plot(x="Unemployment_Rate", y="Stock_index_Price", kind="line")
plt.show()

# نمودار میله‌ای
df.plot(x="Country", y="GDP_per_Country", kind="bar")
plt.show()

# نمودار دایره‌ای
df.plot.pie(
    y="Tasks",
    autopct="%1.1f%%",
    startangle=90,
    legend=False,
    shadow=True,
    explode=(0.1, 0, 0)
)
plt.ylabel("")
plt.show()

### نمودار میله‌ای پشته‌ای (Stacked Bar) بر اساس گروه‌بندی

In [ ]:
a = df.groupby(["DATE", "TYPE"])["SALES"].sum().unstack().fillna(0)
a.plot(kind="bar", stacked=True)
plt.show()

### نمودار پراکندگی (Scatter) روی داده محاسبه‌شده

In [ ]:
df.query("`sepal length (cm)` > 5").assign(
    sepal_ratio=lambda x: x["sepal width (cm)"] / x["sepal length (cm)"]
).plot(kind="scatter", x="sepal_ratio", y="petal_ratio")

---

## 📌 جمع‌بندی سریع

| کار | متد پیشنهادی |
|---|---|
| خواندن فایل | `pd.read_csv()`, `pd.read_excel()` |
| انتخاب سطر/ستون | `df.loc[]`, `df.iloc[]` |
| فیلتر شرطی | `df[df["col"] > x]`, `df.query()` |
| پر کردن داده گمشده | `fillna()`, `interpolate()` |
| ستون جدید | `df["c"] = ...`, `df.assign()` |
| گروه‌بندی | `df.groupby()` |
| تغییر شکل | `pivot_table()`, `melt()` |
| ترکیب جداول | `merge()`, `join()`, `concat()` |
| کار با تاریخ | `pd.to_datetime()`, `.dt`, `.resample()` |
| کار با متن | `.str` |
| رسم نمودار | `.plot()` |

---

*این راهنما بر اساس تمرین‌های عملی روی دیتاست‌های Iris، Titanic و داده‌های آب‌وهوا تهیه شده است.*